# 1.4 — Flow Matching

**Mechanism of the day:** stop building an elaborate noise schedule and just draw a
**straight line** from noise to data. Regress the velocity along that line. Sample by
solving an ODE.

Rung 1.3 worked, and it ended with one real complaint: **200 network evaluations to
produce one batch of samples.** That cost is not incidental. It comes from the fact
that the DDPM reverse process is a *stochastic* walk down a curved path, and curved
paths need small steps.

Flow matching asks the obvious question: what if the path were straight?

**The setup.** Pick a data point `x0` and a noise point `x1 ~ N(0, I)`, and define
the path between them to be a literal straight line:

```
x_t = (1 - t) * x0  +  t * x1          t in [0, 1]     (t=0 data, t=1 noise)
```

Differentiate it. The velocity along that path is

```
u = dx_t/dt = x1 - x0
```

Look at what just happened: the target **does not depend on `t` at all**. There is no
schedule, no `abar`, no posterior variance, no reweighting. A straight line has
constant velocity, and that constant is your regression target.

**The objective** (Conditional Flow Matching):

```
loss = || v_theta(x_t, t)  -  (x1 - x0) ||^2
```

**Sampling** is then an ODE, not a stochastic chain. Start at noise and integrate
`dx/dt = v(x, t)` from `t = 1` down to `t = 0`. Euler steps are enough, and because
the field is close to straight you can take **big** ones.

The subtlety worth stating up front: each *conditional* path is exactly straight, but
the field the network learns is the **average** over all the pairs that pass through a
given point, and that average bends. We will measure exactly how much it bends
(spoiler: ~16%), which is why 2-step sampling still fails and 32-step sampling is
excellent.

We keep 1.3's Ramachandran data and network **identical**, so the comparison at the
end is honest: same data, same architecture, same training budget — only the
objective and the sampler change.

**How to use this notebook:** implement the reps, make the checkpoints pass.
Solutions at the bottom. Trains both models in ~20s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

# ---- identical data to 1.3 ----
BASINS = [('alpha-helix', (-63.0, -43.0), (12.0, 12.0), 0.45),
          ('beta-sheet',  (-120.0, 130.0), (25.0, 22.0), 0.45),
          ('left-handed', (57.0, 40.0),   (12.0, 12.0), 0.10)]
SCALE = 180.0

def make_data(n):
    w = np.array([b[3] for b in BASINS])
    which = rng.choice(len(BASINS), size=n, p=w)
    out = np.zeros((n, 2))
    for i, (_, mu, sd, _) in enumerate(BASINS):
        m = which == i; k = int(m.sum())
        out[m, 0] = rng.normal(mu[0], sd[0], k)
        out[m, 1] = rng.normal(mu[1], sd[1], k)
    return out.astype(np.float32), which

raw, lab = make_data(8000)
X0 = torch.from_numpy(raw / SCALE)

def rama(ax, pts_deg, title, color=BLUE, s=3, alpha=.2):
    ax.scatter(pts_deg[:, 0], pts_deg[:, 1], s=s, alpha=alpha, color=color, lw=0)
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
    ax.set_xlabel('phi'); ax.set_ylabel('psi'); ax.set_title(title, fontsize=10)
    ax.grid(alpha=.15)

# ---- identical network to 1.3 ----
def timestep_embedding(t, dim=64):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32) / half)
    a = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(a), torch.sin(a)], dim=-1)

class Net(nn.Module):
    '''(x, t) -> 2-vector. In 1.3 it predicted noise; here it predicts velocity.'''
    def __init__(self, d=128, tdim=64):
        super().__init__(); self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, d), nn.SiLU(), nn.Linear(d, d))
        self.inp = nn.Linear(2, d)
        self.h1 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.h2 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.out = nn.Sequential(nn.SiLU(), nn.Linear(d, 2))
    def forward(self, x, t):
        h = self.inp(x) + self.tmlp(timestep_embedding(t, self.tdim))
        h = h + self.h1(h); h = h + self.h2(h)
        return self.out(h)

print('data', tuple(X0.shape), '| net params', sum(p.numel() for p in Net().parameters()))
print('convention: t = 0 is DATA, t = 1 is NOISE')

## Part 1 — the path and its velocity

Two functions, four lines of code, and they contain the entire method.

Note how much machinery just disappeared compared with 1.3: no `betas`, no
`cumprod`, no `sqrt(abar)` blending, no posterior variance. Since we feed `t` to the
same sinusoidal embedding as before (which expects a number of order hundreds, not
a fraction), we pass `t * 1000` — a scaling detail, not a concept.

### Rep 1 — `sample_path(x0, x1, t)`
Linear interpolation: `(1 - t) * x0 + t * x1`. `x0`, `x1` are `[B, 2]` and `t` is
`[B]`, so give `t` a trailing axis before multiplying.

In [ ]:
def sample_path(x0, x1, t):
    '''Point at time t on the straight line from data x0 to noise x1.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
a, b = X0[:100], torch.randn(100, 2)
assert torch.allclose(sample_path(a, b, torch.zeros(100)), a), 't=0 must be the data'
assert torch.allclose(sample_path(a, b, torch.ones(100)), b), 't=1 must be the noise'
mid = sample_path(a, b, torch.full((100,), 0.5))
assert torch.allclose(mid, (a + b) / 2, atol=1e-6), 't=0.5 is the midpoint'
print('straight-line path ✓')

fig, ax = plt.subplots(figsize=(4.4, 4.2))
i = torch.randperm(len(X0))[:60]
d, n = X0[i] * SCALE, torch.randn(60, 2) * SCALE
for k in range(60):
    ax.plot([d[k, 0], n[k, 0]], [d[k, 1], n[k, 1]], color=INK, lw=.5, alpha=.35)
ax.scatter(d[:, 0], d[:, 1], s=14, color=GREEN, lw=0, label='data (t=0)', zorder=3)
ax.scatter(n[:, 0], n[:, 1], s=14, color=BLUE, lw=0, label='noise (t=1)', zorder=3)
ax.set_xlabel('phi'); ax.set_ylabel('psi'); ax.grid(alpha=.15)
ax.set_title('every training pair is joined by a straight line')
ax.legend(frameon=False, fontsize=9)
plt.show()

### Rep 2 — `fm_target(x0, x1)`
The velocity along that line. One line of code — and the checkpoint makes the point
that matters: **it is the same at every `t`**.

In [ ]:
def fm_target(x0, x1):
    '''Constant velocity of the straight path from x0 to x1.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
u = fm_target(a, b)
assert u.shape == a.shape
# numerically differentiate the path at two different times: same answer both times
h = 0.01   # the path is exactly linear, so any h gives the exact derivative
for t_probe in (0.15, 0.80):
    t1 = torch.full((100,), t_probe)
    num = (sample_path(a, b, t1 + h) - sample_path(a, b, t1 - h)) / (2 * h)
    assert torch.allclose(num, u, atol=1e-3), 'velocity must equal d(path)/dt'
print('velocity = x1 - x0, identical at t=0.15 and t=0.80 ✓')
print('No schedule, no reweighting: a straight line has constant speed.')

## Part 2 — training

Sample a data point, sample a noise point, sample a random `t`, place yourself on the
line, and regress the velocity. Compare this with `ddpm_loss` from 1.3 — it is the
same shape of computation, with the schedule algebra deleted.

One thing to expect: **this loss plateaus high (~0.45), and that is correct.** The
network sees a point `x_t` that many different `(x0, x1)` pairs could have produced,
and the best it can do is predict the *average* of their velocities. The irreducible
spread of those velocities is the floor. As in 1.3, the loss value is not a quality
score.

### Rep 3 — `fm_loss(model, x0)`
Draw `x1 ~ N(0, I)` and `t ~ U[0, 1)`, build `x_t`, and return the MSE between
`model(x_t, t * 1000)` and the target velocity.

In [ ]:
def fm_loss(model, x0):
    '''Conditional flow-matching loss.'''
    # YOUR CODE HERE
    # hint: torch.rand(len(x0)) for t; remember the t * 1000 scaling for the embedding
    raise NotImplementedError

# --- checkpoint ---
fm = Net()
l = fm_loss(fm, X0[:512])
assert l.dim() == 0 and l.requires_grad, 'need a differentiable scalar'
assert 0.3 < l.item() < 3.0, 'untrained loss should be order 1'
print('untrained flow-matching loss %.3f ✓' % l.item())

In [ ]:
opt = torch.optim.AdamW(fm.parameters(), lr=2e-3)
hist, t0 = [], time.time()
for step in range(1, 5001):
    ix = torch.randint(0, len(X0), (256,))
    loss = fm_loss(fm, X0[ix])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0 or step == 1:
        hist.append((step, loss.item()))
        if step % 2500 == 0 or step == 1:
            print('step %4d  loss %.4f' % (step, loss.item()))
print('flow model trained in %.1fs' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2)
ax.set_xlabel('step'); ax.set_ylabel('MSE on velocity'); ax.grid(alpha=.15)
ax.set_title('regressing a velocity field')
plt.show()

## Part 3 — sampling is now an ODE

No noise, no posterior variance, no ancestral chain. Start from `x ~ N(0, I)` at
`t = 1` and walk downhill in time:

```
x  <-  x  -  dt * v_theta(x, t)          t: 1 -> 0
```

The minus sign is because we integrate *backwards* in time (from noise at `t=1`
toward data at `t=0`) while `v` is defined as `dx/dt`.

This is plain **Euler integration**, the simplest ODE solver there is. `steps` is now
a free parameter you choose at sampling time — the model does not care. That is the
whole prize: in 1.3, `T = 200` was baked into training.

### Rep 4 — `ode_sample(model, n, steps, keep_traj=False)`
Euler-integrate from `t=1` to `t=0` in `steps` equal increments. Return the final
`[n, 2]` points, and (optionally) the list of intermediate states so we can look at
the trajectories.

In [ ]:
@torch.no_grad()
def ode_sample(model, n, steps, keep_traj=False):
    '''Euler-integrate the learned velocity field from noise to data.'''
    # YOUR CODE HERE
    # hint: dt = 1.0 / steps; for i in range(steps): t = 1.0 - i * dt
    #       v = model(x, torch.full((n,), t * 1000)); x = x - dt * v
    raise NotImplementedError

# --- checkpoint ---
s32 = ode_sample(fm, 2000, 32)
assert s32.shape == (2000, 2)
assert s32.abs().max() < 3.0, 'samples should land near the data'
x_, traj = ode_sample(fm, 8, 16, keep_traj=True)
assert len(traj) == 17, 'keep_traj should record the start plus one state per step'
print('ODE sampling ✓ — 32 steps instead of 1.3\'s 200')

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.5))
for ax, k in zip(axes, [2, 4, 8, 32]):
    rama(ax, ode_sample(fm, 4000, k).numpy() * SCALE, '%d Euler steps' % k)
plt.tight_layout(); plt.show()
print('At 2 steps the straight-line approximation is too crude; by 8 it is recognisable;')
print('by 32 it is done. Now let us measure that instead of eyeballing it.')

## Part 4 — measuring quality (so the comparison is not vibes)

Scatter plots are easy to fool yourself with. We need a number for "how close is this
cloud to the real distribution."

We use **total-variation distance between 2-D histograms**: bin both point sets on
the same grid, and compute `0.5 * sum |p - q|`. It is `0` for identical distributions
and `1` for disjoint ones.

Crucially, we also compute the **noise floor**: the TVD between two independent draws
of the *real* data. Finite samples never give exactly `0`, so that floor is the best
score any model could achieve here. Without it, a TVD of 0.19 is meaningless.

### Rep 5 — `tvd(points_deg)`
Histogram `points_deg` on the shared `BINS` grid, normalize to sum to 1, and return
half the absolute difference from the precomputed reference histogram `REF`.

In [ ]:
BINS = np.linspace(-180, 180, 31)

def hist2d(p):
    h, _, _ = np.histogram2d(p[:, 0], p[:, 1], bins=[BINS, BINS])
    return h / h.sum()

REF = hist2d(raw)          # reference: the real data

def tvd(points_deg):
    '''Total-variation distance to the real data histogram. 0 = identical.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
assert abs(tvd(raw)) < 1e-9, 'the data must score exactly 0 against itself'
held_out, _ = make_data(8000)
FLOOR = tvd(held_out)
assert 0.0 < FLOOR < 0.2, 'the finite-sample floor should be small but nonzero'
assert tvd(rng.uniform(-180, 180, (8000, 2))) > 0.7, 'uniform noise should score terribly'
print('TVD: real vs itself %.4f | real vs held-out real %.4f (the floor) | uniform %.3f'
      % (tvd(raw), FLOOR, tvd(rng.uniform(-180, 180, (8000, 2)))))
print('No model can beat %.4f on this metric with 8000 samples ✓' % FLOOR)

### The head-to-head

Now we train a DDPM on the identical data with the identical network, and sample it
with **DDIM-style striding** — the standard way to run a diffusion model in fewer
steps (take the same trained model, but skip timesteps and use the deterministic
update). This is the fair comparison: both samplers are deterministic, both use one
network evaluation per step, and both models saw the same training budget.

The DDPM code below is given, not a rep — you already wrote all of it in 1.3.

In [ ]:
# ---- 1.3's DDPM, verbatim: schedule, forward jump, loss ----
T = 200
betas = torch.linspace(1e-4, 0.06, T); alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

def q_sample(x0, t, noise):
    a = abar[t].unsqueeze(-1)
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

def ddpm_loss(model, x0):
    t = torch.randint(0, T, (len(x0),)); noise = torch.randn_like(x0)
    return F.mse_loss(model(q_sample(x0, t, noise), t), noise)

dd = Net()
opt = torch.optim.AdamW(dd.parameters(), lr=2e-3)
t0 = time.time()
for step in range(1, 5001):
    ix = torch.randint(0, len(X0), (256,))
    loss = ddpm_loss(dd, X0[ix])
    opt.zero_grad(); loss.backward(); opt.step()
print('DDPM baseline trained in %.1fs (same net, same budget)' % (time.time() - t0))

@torch.no_grad()
def ddim_sample(model, n, steps):
    '''Deterministic strided sampling of a trained DDPM.'''
    x = torch.randn(n, 2)
    ts = np.linspace(T - 1, 0, steps).round().astype(int)
    for i, t in enumerate(ts):
        eps = model(x, torch.full((n,), int(t), dtype=torch.long))
        a_t = abar[t]
        x0_hat = (x - (1 - a_t).sqrt() * eps) / a_t.sqrt()      # implied clean point
        if i == len(ts) - 1:
            x = x0_hat
        else:
            a_prev = abar[ts[i + 1]]
            x = a_prev.sqrt() * x0_hat + (1 - a_prev).sqrt() * eps
    return x

NFE = [2, 4, 8, 16, 32, 64, 200]
fm_tvd, dd_tvd = [], []
for k in NFE:
    fm_tvd.append(tvd(ode_sample(fm, 8000, k).numpy() * SCALE))
    dd_tvd.append(tvd(ddim_sample(dd, 8000, k).numpy() * SCALE))

print('\n NFE   flow matching   DDPM')
for k, a_, b_ in zip(NFE, fm_tvd, dd_tvd):
    print('%4d      %.4f       %.4f' % (k, a_, b_))

fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(NFE, fm_tvd, '-o', color=BLUE, lw=2, ms=6, label='flow matching')
ax.plot(NFE, dd_tvd, '-o', color=GREEN, lw=2, ms=6, label='DDPM (DDIM striding)')
ax.axhline(FLOOR, color=INK, ls='--', lw=1.5)
ax.text(200, FLOOR, 'finite-sample floor', ha='right', va='bottom', fontsize=9, color=INK)
ax.set_xscale('log'); ax.set_xticks(NFE); ax.set_xticklabels(NFE)
ax.set_xlabel('network evaluations per sample (NFE)')
ax.set_ylabel('TVD to real data  (lower is better)')
ax.set_title('the same quality, far cheaper')
ax.grid(alpha=.15); ax.legend(frameon=False, fontsize=9)
plt.show()
print('Flow matching reaches DDPM\'s 200-step quality in roughly 32 steps, and it is')
print('already usable at 8 — where the diffusion sampler is still badly broken. ✓')

## Part 5 — how straight is it *really*?

The pitch was "straight paths, so take big steps." If that were literally true of the
learned field, **one** Euler step would be exact — and we saw that 2 steps produce
garbage. So it is not literally true, and it is worth understanding why.

Each *conditional* path (one `x0`, one `x1`) is exactly straight. But the network
cannot see which pair it is on; at a point `x_t` it must output the **average**
velocity over every pair passing through there. Averaging straight lines that point
in different directions gives a field whose integral curves **bend**.

So let us measure the bend: sample trajectories, and for each one compare the actual
path against the straight line joining its own endpoints.

### Rep 6 — `straightness(traj)`
`traj` is a list of `[n, 2]` tensors from `keep_traj=True`. For each intermediate
step, compare the true position with the linear interpolation between the first and
last states. Return the **mean deviation as a fraction of the endpoint distance**
(so `0.0` is perfectly straight).

In [ ]:
def straightness(traj):
    '''Mean deviation from the straight line between endpoints, relative to its length.'''
    # YOUR CODE HERE
    # hint: tr = torch.stack(traj); start, end = tr[0], tr[-1]
    #       for step i of N: lin = start + (i/N) * (end - start)
    #       accumulate (tr[i] - lin).norm(dim=-1).mean(), divide by (end-start).norm(dim=-1).mean()
    raise NotImplementedError

# --- checkpoint ---
perfect = [torch.zeros(5, 2) + torch.tensor([float(i), 0.0]) for i in range(9)]
assert straightness(perfect) < 1e-6, 'a genuinely straight path must score 0'
_, tr = ode_sample(fm, 2000, 64, keep_traj=True)
bend = straightness(tr)
assert 0.0 < bend < 0.6, 'learned paths bend, but not wildly'
print('learned trajectories deviate %.1f%% from perfectly straight' % (100 * bend))
print('-> big steps work well, but not arbitrarily big. Hence 2 steps failed. ✓')

fig, ax = plt.subplots(figsize=(4.4, 4.2))
_, tr = ode_sample(fm, 40, 64, keep_traj=True)
P = torch.stack(tr).numpy() * SCALE
for k in range(P.shape[1]):
    ax.plot(P[:, k, 0], P[:, k, 1], color=BLUE, lw=.8, alpha=.6)
ax.scatter(P[0, :, 0], P[0, :, 1], s=16, color=INK, lw=0, zorder=3, label='start (noise)')
ax.scatter(P[-1, :, 0], P[-1, :, 1], s=16, color=GREEN, lw=0, zorder=3, label='end (data)')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel('phi'); ax.set_ylabel('psi'); ax.grid(alpha=.15)
ax.set_title('sampling trajectories: nearly straight, gently bent')
ax.legend(frameon=False, fontsize=9)
plt.show()

## Reflection — what just transferred

- **Same destination, simpler road.** Diffusion and flow matching both learn to turn
  noise into data. Flow matching gets there by regressing a velocity along a straight
  interpolation, which deletes the schedule algebra entirely. Fewer moving parts means
  fewer things to get subtly wrong.
- **The target is `x1 - x0`, independent of `t`.** That is the whole reason the method
  feels so light compared with `sqrt(abar)` bookkeeping.
- **Sampling cost became a dial, not a constant.** `T` was frozen at training time in
  1.3; `steps` is chosen at sampling time here. Same weights, 8 steps or 200.
- **Straightness is approximate, and you should measure it.** The conditional paths
  are straight; the learned average field bends by ~16%, which is exactly why 2 steps
  fail. (Methods like rectified flow / reflow iterate to straighten this further.)
- **Always compute the floor.** Two draws of the real data score 0.09 on our metric,
  not 0. A model at 0.19 is decent; without the floor you cannot tell.
- **This is why the field moved.** Flow matching underpins Stable Diffusion 3 and the
  newer protein generators, and the *same* straight-path idea generalizes to curved
  spaces — which is precisely what rung 1.6 needs for SE(3) frames.

**Next rep:** `1.5 Discrete / masked diffusion` — bring diffusion back to sequences.
It resolves the loose end from 1.2: instead of Gibbs-sampling a masked model and
hoping, train under a schedule of corruption levels so iterative unmasking becomes a
principled generative process. This is the core of DPLM.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def sample_path(x0, x1, t):
    tt = t.unsqueeze(-1)                       # [B, 1] to broadcast over [B, 2]
    return (1 - tt) * x0 + tt * x1

def fm_target(x0, x1):
    return x1 - x0                             # constant in t — that is the point

def fm_loss(model, x0):
    x1 = torch.randn_like(x0)
    t = torch.rand(len(x0))
    xt = sample_path(x0, x1, t)
    return F.mse_loss(model(xt, t * 1000), fm_target(x0, x1))

@torch.no_grad()
def ode_sample(model, n, steps, keep_traj=False):
    x = torch.randn(n, 2)
    dt = 1.0 / steps
    traj = [x.clone()]
    for i in range(steps):
        t = 1.0 - i * dt                       # walk t from 1 down to 0
        v = model(x, torch.full((n,), t * 1000))
        x = x - dt * v
        if keep_traj:
            traj.append(x.clone())
    return (x, traj) if keep_traj else x

def tvd(points_deg):
    return 0.5 * np.abs(hist2d(points_deg) - REF).sum()

def straightness(traj):
    tr = torch.stack(traj)
    start, end = tr[0], tr[-1]
    span = (end - start).norm(dim=-1).mean()
    n = len(tr) - 1
    devs = []
    for i in range(1, n):
        lin = start + (i / n) * (end - start)
        devs.append((tr[i] - lin).norm(dim=-1).mean())
    if not devs:
        return 0.0
    return float(torch.stack(devs).mean() / span)

print('reference solutions loaded — re-run the checkpoint cells above')